## Training Random Forest + Ekspor TFLite — URL Classifier
**Google Colab | Multi-Dataset | Export ke Android**

---
Notebook ini melatih model Random Forest dan mengekspornya ke format TFLite
yang kompatibel dengan aplikasi Android, untuk setiap ukuran dataset secara berurutan.

**Alur kerja:**
1. Load CSV per ukuran dataset dari `DATASET_FILES`
2. Sampling balanced + validasi konsistensi fitur (F5, F7)
3. Training Random Forest dengan hyperparameter terbaik dari tuning
4. Evaluasi (accuracy, F1, AUC-ROC, confusion matrix)
5. Ekspor ke TFLite via pipeline: **sklearn → ONNX → TFLite**
6. Verifikasi TFLite pada seluruh test set
7. Simpan semua output ke subfolder Drive per ukuran dataset

**Catatan konversi:**
- `zipmap=False` saat konversi ONNX → **menghindari error GATHER di Android**
- Fungsi ekstraksi fitur **identik** dengan kode Android (`RandomForestClassifier.kt`)

## Cell 1 — Install Library

Instalasi `skl2onnx` dan `onnx2tf` untuk pipeline konversi ke TFLite.

In [ ]:
# Library standar ML
!pip install -q --upgrade scikit-learn seaborn

# Pipeline konversi: sklearn → ONNX
# skl2onnx: konverter resmi sklearn ke ONNX
!pip install -q skl2onnx onnx onnxruntime

# Pipeline konversi: ONNX → TFLite
# onnx2tf: library aktif yang menangani banyak ONNX op termasuk RF
!pip install -q onnx2tf

print('Semua library berhasil diinstall.')

## Cell 2 — Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import time
import os
import re
import shutil
import subprocess
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix,
    roc_curve, auc
)

import tensorflow as tf
import onnx
import onnxruntime as ort
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

print(f'scikit-learn : {__import__("sklearn").__version__}')
print(f'TensorFlow   : {tf.__version__}')
print(f'ONNX         : {onnx.__version__}')

## Cell 3 — Mount Google Drive & Konfigurasi

**Format dataset** — CSV dengan kolom minimal:
```
label, domain_length, digit_count, dot_count, delimiter_count,
suspicious_word_count, digit_letter_ratio, max_sequential_digits
```

> `label`: 0 = URL aman, 1 = URL pornografi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ================================================================
# SESUAIKAN PATH INI
# ================================================================
DATASET_BASE     = '/content/drive/MyDrive/Tugas Akhir/Dataset/'
SAVE_PATH        = '/content/drive/MyDrive/Tugas Akhir/Training/RF/'
BEST_PARAMS_JSON = '/content/drive/MyDrive/Tugas Akhir/Tuning/RF/best_params.json'
LABEL_COL        = 'label'

FEATURE_COLS = [
    'domain_length',
    'digit_count',
    'dot_count',
    'delimiter_count',
    'suspicious_word_count',   # F5: COUNT, bukan boolean
    'digit_letter_ratio',
    'max_sequential_digits'    # F7: panjang MAX, bukan boolean
]

# Daftar file CSV - setiap file = satu ukuran dataset terpisah
# SIZE_LABEL otomatis dari nama file (tanpa ekstensi)
DATASET_FILES = [
    DATASET_BASE + 'dataset_100rb.csv',
    DATASET_BASE + 'dataset_200rb.csv',
    DATASET_BASE + 'dataset_300rb.csv',
]
# ================================================================

os.makedirs(SAVE_PATH, exist_ok=True)
print('Dataset files:')
for fpath in DATASET_FILES:
    size_label = os.path.splitext(os.path.basename(fpath))[0]
    ok = os.path.exists(fpath)
    if ok:
        tmp = pd.read_csv(fpath, usecols=[LABEL_COL])
        n0 = int((tmp[LABEL_COL] == 0).sum())
        n1 = int((tmp[LABEL_COL] == 1).sum())
        print(f'  {size_label}: OK  {len(tmp):,} baris | aman={n0:,} | porno={n1:,}')
    else:
        print(f'  {size_label}: FILE TIDAK ADA - upload dulu ke Drive')

## Cell 4 — Fungsi Ekstraksi Fitur

> **PENTING**: Fungsi ini HARUS IDENTIK dengan `extractLexicalFeatures()` di `RandomForestClassifier.kt`.
> Jika ada perbedaan sekecil apapun, model akan salah mengklasifikasi di Android.

Digunakan untuk:
1. Verifikasi bahwa fitur di dataset konsisten dengan cara hitung Android
2. Mengekstrak fitur dari URL baru jika dataset hanya berisi URL+label

**Perbedaan dengan training sebelumnya:**
- F5 `suspicious_word_count`: sekarang **COUNT** (bisa 0,1,2,...), bukan boolean
- F7 `max_sequential_digits`: sekarang **panjang maksimal** digit berurutan, bukan boolean

In [ ]:
# Daftar kata mencurigakan — HARUS SAMA dengan SUSPICIOUS_WORDS di Android
SUSPICIOUS_WORDS = {
    'porn', 'sex', 'xxx', 'adult', 'cam', 'tube', 'bokep', 'hentai',
    'nude', 'gay', 'lesbian', 'erotic', 'mature', 'amateur',
    'creampie', 'milf', 'bbw', 'naked', 'porno', 'anal',
    'pussy', 'cock', 'cumshot', 'orgasm', 'xvideos', 'xnxx',
    'xhamster', 'redtube', 'youporn', 'brazzers', 'onlyfans'
}

def extract_features(domain: str) -> list:
    """
    Ekstrak 7 fitur leksikal dari domain URL.
    Input : 'youporn.com' (domain setelah normalisasi, dengan TLD)
    Output: [F1, F2, F3, F4, F5, F6, F7]
    """
    lower = domain.lower()

    digits  = sum(c.isdigit() for c in lower)           # F2
    letters = sum(c.isalpha() for c in lower)           # untuk F6

    # F5: COUNT kata mencurigakan (identik dgn SUSPICIOUS_WORDS.count{} di Android)
    suspicious_word_count = sum(1 for w in SUSPICIOUS_WORDS if w in lower)

    # F6: digit / letter ratio
    digit_letter_ratio = digits / letters if letters > 0 else float(digits)

    # F7: panjang maksimal urutan digit (identik dgn Regex("\\d+").findAll().maxOfOrNull di Android)
    sequences = re.findall(r'\d+', lower)
    max_sequential_digits = max((len(s) for s in sequences), default=0)

    return [
        len(domain),                                                    # F1: domain_length
        digits,                                                         # F2: digit_count
        lower.count('.'),                                               # F3: dot_count
        sum(1 for c in lower if not c.isalnum() and c != '.'),         # F4: delimiter_count
        suspicious_word_count,                                          # F5: suspicious_word_count
        digit_letter_ratio,                                             # F6: digit_letter_ratio
        max_sequential_digits,                                          # F7: max_sequential_digits
    ]

# Verifikasi dengan contoh konkret — cocokkan dengan logcat Android
test_cases = [
    ('youporn.com',       1, 'F5=1(porn), F7=0'),
    ('xvideos.com',       1, 'F5=1(xvideos), F7=0'),
    ('xxx18hub.com',      1, 'F5=1(xxx), F7=2(dari 18)'),
    ('bba021.com',        0, 'F5=0, F7=3(dari 021)'),
    ('google.com',        0, 'semua fitur rendah'),
    ('hand-job-porn.com', 1, 'F5=1(porn), F4=1(dash), F7=0'),
]

print(f'{"Domain":<25} {"Lbl":<5} {"F1":>4} {"F2":>4} {"F3":>4} {"F4":>4} {"F5":>4} {"F6":>7} {"F7":>4}   Catatan')
print('-' * 95)
for domain, label, note in test_cases:
    f = extract_features(domain)
    print(f'{domain:<25} {label:<5} {f[0]:>4} {f[1]:>4} {f[2]:>4} {f[3]:>4} {f[4]:>4} {f[5]:>7.3f} {f[6]:>4}   {note}')

## Cell 5 — Load Hyperparameter Terbaik

Notebook akan mencoba membaca `best_params.json` dari hasil tuning.
Jika tidak ada, gunakan nilai default.

In [ ]:
DEFAULT_PARAMS = {
    'n_estimators': 200,
    'max_depth'   : 10,
    'max_features': 'sqrt',
}

if os.path.exists(BEST_PARAMS_JSON):
    with open(BEST_PARAMS_JSON) as f:
        best_params = json.load(f)
    print(f'Loaded dari: {BEST_PARAMS_JSON}')
else:
    best_params = DEFAULT_PARAMS
    print('best_params.json tidak ditemukan.')
    print('   Menggunakan default hyperparameter.')
    print('   Jalankan rf_hyperparameter_tuning_v2.ipynb untuk tuning.')

print('Hyperparameter yang akan digunakan:')
for k, v in best_params.items():
    print(f'  {k:<22}: {v}')

## Cell 6 — Training Loop (Multi-Dataset)

Loop ini memproses setiap file CSV di `DATASET_FILES` secara berurutan:
1. Load & balanced sampling per kelas
2. Validasi konsistensi fitur F5 & F7
3. Split 80/20 → training Random Forest
4. Evaluasi (accuracy, F1, AUC-ROC, confusion matrix + distribusi skor)
5. Feature importance plot
6. Cek overfitting (train vs test accuracy)
7. Konversi ke TFLite (sklearn → ONNX → TFLite) + verifikasi pada test set
8. Simpan semua output ke subfolder Drive per ukuran dataset

Estimasi waktu per dataset (CPU Colab): **1–5 menit**

In [ ]:
# Set True hanya jika F5/F7 masih boolean di dataset (akan rekalkulasi dari kolom 'url')
RECOMPUTE_FEATURES = False

all_results = {}

for csv_path in DATASET_FILES:
    SIZE_LABEL = os.path.splitext(os.path.basename(csv_path))[0]

    # ── 1. Load CSV ───────────────────────────────────────────────
    df = pd.read_csv(csv_path)
    n_aman  = int((df[LABEL_COL] == 0).sum())
    n_porno = int((df[LABEL_COL] == 1).sum())
    N_PER_CLASS = min(n_aman, n_porno)

    print(f'\n{"="*65}')
    print(f'  {SIZE_LABEL}  ({len(df):,} baris | {N_PER_CLASS:,}/kelas)')
    print(f'{"="*65}')

    SIZE_SAVE_PATH = SAVE_PATH + f'{SIZE_LABEL}/'
    os.makedirs(SIZE_SAVE_PATH, exist_ok=True)

    # ── 2. Sampling balanced ──────────────────────────────────────
    df_safe = df[df[LABEL_COL] == 0].sample(n=N_PER_CLASS, random_state=42)
    df_porn = df[df[LABEL_COL] == 1].sample(n=N_PER_CLASS, random_state=42)
    df_bal  = pd.concat([df_safe, df_porn]).sample(frac=1, random_state=42).reset_index(drop=True)
    df_bal  = df_bal.drop_duplicates(subset=FEATURE_COLS + [LABEL_COL])
    df_bal  = df_bal.dropna(subset=FEATURE_COLS + [LABEL_COL]).reset_index(drop=True)
    print(f'Setelah sampling & dedup: {len(df_bal):,} baris')

    # ── 3. (Opsional) Hitung ulang fitur dari URL ─────────────────
    if RECOMPUTE_FEATURES:
        if 'url' not in df_bal.columns:
            print('  Kolom url tidak ditemukan, lewati rekalkulasi.')
        else:
            print('  Menghitung ulang fitur dari URL...')
            t0 = time.time()
            feat_list = df_bal['url'].apply(extract_features).tolist()
            feat_df   = pd.DataFrame(feat_list, columns=FEATURE_COLS)
            for col in FEATURE_COLS:
                df_bal[col] = feat_df[col].values
            print(f'  Selesai dalam {time.time()-t0:.1f} detik')

    # ── 4. Validasi F5 & F7 ───────────────────────────────────────
    f5_ok = not (set(df_bal['suspicious_word_count'].unique()) <= {0, 1})
    f7_ok = not (set(df_bal['max_sequential_digits'].unique()) <= {0, 1})
    print(f'F5 (int count): {"OK" if f5_ok else "PERHATIAN: masih boolean, set RECOMPUTE_FEATURES=True"}')
    print(f'F7 (max len)  : {"OK" if f7_ok else "PERHATIAN: masih boolean, set RECOMPUTE_FEATURES=True"}')

    # ── 5. Persiapan data (80/20 split) ──────────────────────────
    X = df_bal[FEATURE_COLS].values.astype(np.float32)
    y = df_bal[LABEL_COL].values
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')

    # ── 6. Training ───────────────────────────────────────────────
    rf_model = RandomForestClassifier(**best_params, random_state=42, n_jobs=-1)
    print(f'\nTraining {SIZE_LABEL}...')
    t0 = time.time()
    rf_model.fit(X_train, y_train)
    durasi = time.time() - t0
    print(f'Selesai: {durasi:.1f} detik | {len(rf_model.estimators_)} pohon | max_depth={rf_model.max_depth}')

    # ── 7. Evaluasi Test Set ──────────────────────────────────────
    y_pred      = rf_model.predict(X_test)
    y_pred_prob = rf_model.predict_proba(X_test)[:, 1]
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    f1   = f1_score(y_test, y_pred, zero_division=0)
    fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
    roc_auc = auc(fpr, tpr)

    print(f'\nTest Set — {SIZE_LABEL}:')
    print(f'  Accuracy : {acc:.4f}  Precision: {prec:.4f}')
    print(f'  Recall   : {rec:.4f}  F1-Score : {f1:.4f}  AUC: {roc_auc:.4f}')
    print(classification_report(y_test, y_pred,
          target_names=['Aman (0)', 'Pornografi (1)'], digits=4))

    # ── 8. Confusion Matrix + Distribusi Skor ────────────────────
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(f'Evaluasi — RF {SIZE_LABEL}', fontsize=12)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Pred: Aman', 'Pred: Porno'],
                yticklabels=['Actual: Aman', 'Actual: Porno'], ax=axes[0])
    axes[0].set_title('Confusion Matrix')
    axes[1].hist(y_pred_prob[y_test == 0], bins=50, alpha=0.6,
                 label='Aman', color='green', density=True)
    axes[1].hist(y_pred_prob[y_test == 1], bins=50, alpha=0.6,
                 label='Porno', color='red', density=True)
    axes[1].axvline(0.5, color='black', linestyle='--', label='Thr=0.5')
    axes[1].set_title('Distribusi Skor Probabilitas')
    axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'/content/evaluation_{SIZE_LABEL}.png', dpi=120, bbox_inches='tight')
    plt.show(); plt.close()

    # ── 9. Feature Importance ─────────────────────────────────────
    imp_df = pd.DataFrame({
        'Fitur'     : FEATURE_COLS,
        'Importance': rf_model.feature_importances_
    }).sort_values('Importance', ascending=True)
    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.barh(imp_df['Fitur'], imp_df['Importance'], color='#1E88E5', edgecolor='white')
    for bar, val in zip(bars, imp_df['Importance']):
        ax.text(val + 0.002, bar.get_y() + bar.get_height() / 2,
                f'{val:.4f}', va='center', fontsize=9)
    ax.set_title(f'Feature Importance — RF {SIZE_LABEL}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Importance Score'); ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'/content/feature_importance_{SIZE_LABEL}.png', dpi=150, bbox_inches='tight')
    plt.show(); plt.close()

    # ── 10. Cek Overfitting ───────────────────────────────────────
    y_pred_train = rf_model.predict(X_train)
    acc_train = accuracy_score(y_train, y_pred_train)
    gap = acc_train - acc
    print(f'Overfitting: train={acc_train:.4f} | test={acc:.4f} | gap={gap:.4f}', end='  ')
    print('OK' if gap < 0.02 else ('Sedikit overfit' if gap < 0.05 else 'Overfit — kurangi max_depth'))

    # ── 11. Simpan Joblib ─────────────────────────────────────────
    joblib_path = f'/content/rf_model_{SIZE_LABEL}.joblib'
    joblib.dump(rf_model, joblib_path, compress=3)

    # ── 12. sklearn → ONNX ───────────────────────────────────────
    options  = {id(rf_model): {'zipmap': False}}
    onnx_mdl = convert_sklearn(
        rf_model,
        initial_types=[('float_input', FloatTensorType([None, len(FEATURE_COLS)]))],
        options=options, target_opset=11
    )
    # Buang output label, pertahankan output probabilitas saja
    prob_out = next((o for o in onnx_mdl.graph.output if 'prob' in o.name.lower()),
                    onnx_mdl.graph.output[1] if len(onnx_mdl.graph.output) >= 2
                    else onnx_mdl.graph.output[0])
    del onnx_mdl.graph.output[:]
    onnx_mdl.graph.output.append(prob_out)
    onnx_path = f'/content/rf_{SIZE_LABEL}.onnx'
    onnx.save(onnx_mdl, onnx_path)

    # Verifikasi ONNX vs sklearn
    sess     = ort.InferenceSession(onnx_path)
    inp_name = sess.get_inputs()[0].name
    out_name = sess.get_outputs()[0].name
    test_vec = np.array([[10, 0, 1, 0, 1, 0.0, 0]], dtype=np.float32)
    onnx_out = sess.run([out_name], {inp_name: test_vec})[0][0]
    sk_prob  = rf_model.predict_proba(test_vec)[0]
    onnx_delta = max(abs(onnx_out - sk_prob))
    print(f'ONNX delta vs sklearn: {onnx_delta:.6f}', 'OK' if onnx_delta < 0.01 else 'PERHATIAN')

    # ── 13. ONNX → TFLite ────────────────────────────────────────
    tf_dir = f'/content/rf_tf_{SIZE_LABEL}'
    os.makedirs(tf_dir, exist_ok=True)
    result = subprocess.run(
        ['onnx2tf', '-i', onnx_path, '-o', tf_dir, '-osd', '-nuo'],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f'onnx2tf error:\n{result.stderr[-500:]}')
        raise RuntimeError(f'onnx2tf gagal untuk {SIZE_LABEL}')
    converter = tf.lite.TFLiteConverter.from_saved_model(tf_dir)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_model = converter.convert()
    tflite_path  = f'/content/url_classifier_rf_{SIZE_LABEL}.tflite'
    with open(tflite_path, 'wb') as f_out:
        f_out.write(tflite_model)
    print(f'TFLite: {len(tflite_model)/1024:.1f} KB')

    # ── 14. Verifikasi TFLite pada Test Set ──────────────────────
    interp = tf.lite.Interpreter(model_content=tflite_model)
    interp.allocate_tensors()
    inp_det = interp.get_input_details()[0]
    out_det = interp.get_output_details()[0]
    assert inp_det['dtype'] == np.float32, 'Input harus float32'
    assert out_det['shape'][-1] == 2,      'Output harus [1,2]'

    # Verifikasi batch: jalankan semua test sekaligus
    tflite_probs = []
    BATCH_VER = 512
    for start in range(0, len(X_test), BATCH_VER):
        batch = X_test[start:start+BATCH_VER].astype(np.float32)
        for j in range(len(batch)):
            interp.set_tensor(inp_det['index'], batch[j:j+1])
            interp.invoke()
            tflite_probs.append(float(interp.get_tensor(out_det['index'])[0][1]))

    tflite_binary = (np.array(tflite_probs) >= 0.5).astype(int)
    tflite_acc    = accuracy_score(y_test, tflite_binary)
    tflite_f1     = f1_score(y_test, tflite_binary, zero_division=0)
    tflite_prec   = precision_score(y_test, tflite_binary, zero_division=0)
    tflite_rec    = recall_score(y_test, tflite_binary, zero_division=0)
    delta_acc     = abs(acc - tflite_acc)
    print(f'TFLite acc={tflite_acc:.4f} | delta={delta_acc:.4f}', 'OK' if delta_acc < 0.005 else 'PERHATIAN')

    # ── 15. Simpan ke Drive ───────────────────────────────────────
    report_str  = classification_report(y_test, y_pred,
                      target_names=['Aman (0)', 'Pornografi (1)'], digits=4)
    report_path = f'/content/classification_report_{SIZE_LABEL}.txt'
    with open(report_path, 'w') as f_out:
        f_out.write(f'Classification Report — RF {SIZE_LABEL}\n{"="*50}\n\n')
        f_out.write(report_str)
        f_out.write(f'\nAccuracy : {acc:.4f}\nPrecision: {prec:.4f}\n'
                    f'Recall   : {rec:.4f}\nF1-Score : {f1:.4f}\nAUC-ROC  : {roc_auc:.4f}\n')

    for src, fname in [
        (tflite_path,                                      f'url_classifier_rf_{SIZE_LABEL}.tflite'),
        (onnx_path,                                        f'rf_{SIZE_LABEL}.onnx'),
        (joblib_path,                                      f'rf_model_{SIZE_LABEL}.joblib'),
        (f'/content/evaluation_{SIZE_LABEL}.png',         f'evaluation_{SIZE_LABEL}.png'),
        (f'/content/feature_importance_{SIZE_LABEL}.png', f'feature_importance_{SIZE_LABEL}.png'),
        (report_path,                                      f'classification_report_{SIZE_LABEL}.txt'),
    ]:
        if os.path.exists(src):
            shutil.copy(src, SIZE_SAVE_PATH + fname)
            print(f'  OK {fname}')

    # ── 16. Catat hasil ───────────────────────────────────────────
    all_results[SIZE_LABEL] = {
        'csv_file'         : os.path.basename(csv_path),
        'total_rows_csv'   : len(df),
        'n_per_class'      : N_PER_CLASS,
        'total_samples'    : len(df_bal),
        'train_samples'    : len(X_train),
        'test_samples'     : len(X_test),
        'train_time_sec'   : round(durasi, 1),
        'n_estimators'     : rf_model.n_estimators,
        'max_depth'        : str(rf_model.max_depth),
        'accuracy'         : round(acc,         4),
        'precision'        : round(prec,        4),
        'recall'           : round(rec,         4),
        'f1_score'         : round(f1,          4),
        'auc_roc'          : round(roc_auc,     4),
        'train_accuracy'   : round(acc_train,   4),
        'overfit_gap'      : round(gap,         4),
        'tflite_accuracy'  : round(tflite_acc,  4),
        'tflite_f1'        : round(tflite_f1,   4),
        'tflite_prec'      : round(tflite_prec, 4),
        'tflite_rec'       : round(tflite_rec,  4),
        'tflite_delta_acc' : round(delta_acc,   4),
        'tflite_kb'        : round(len(tflite_model) / 1024, 1),
        'tp': int(tp), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn),
        'feature_importance': {col: round(float(imp), 4)
                               for col, imp in zip(FEATURE_COLS, rf_model.feature_importances_)},
    }
    print(f'\nSELESAI {SIZE_LABEL} — acc={acc:.4f} | f1={f1:.4f} | auc={roc_auc:.4f} | {len(tflite_model)/1024:.1f}KB')
    print(f'{"─"*65}')

# ── Simpan semua hasil ke JSON ────────────────────────────────────
results_path = SAVE_PATH + 'rf_all_results.json'
with open(results_path, 'w') as f_out:
    json.dump(all_results, f_out, indent=2)
print(f'\nall_results: {results_path}')

# ── Ringkasan Perbandingan ────────────────────────────────────────
print(f'\n{"="*75}')
print('SEMUA TRAINING SELESAI — RINGKASAN')
print(f'{"="*75}')
print(f'{"Dataset":<20} {"Rows":>8} {"Samples":>10} {"Acc":>7} {"F1":>7} {"AUC":>7} {"TFLite Acc":>11} {"KB":>6}')
print(f'{"─"*75}')
for lbl, r in all_results.items():
    print(f'{lbl:<20} {r["total_rows_csv"]:>8,} {r["total_samples"]:>10,} '
          f'{r["accuracy"]:>7.4f} {r["f1_score"]:>7.4f} {r["auc_roc"]:>7.4f} '
          f'{r["tflite_accuracy"]:>11.4f} {r["tflite_kb"]:>6.1f}')
print(f'{"="*75}')